# Render characters for models A–C

Kannada needs Pillow's RAQM layout engine, which works in Colab but not on the Windows laptop.
Before running: upload the project folder to Drive as `MyDrive/ocr-project`, and build
`charsets/kannada.txt` with `src/build_charset.py`. The zips end up in `data/` on Drive; download
them and unzip into `data/rendered/` locally.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ocr-project

In [ ]:
!apt-get -qq install -y libraqm-dev fonts-noto-core fonts-lohit-knda fonts-dejavu-core fonts-liberation > /dev/null
!pip -q install fonttools pyyaml

In [ ]:
from PIL import features
print('raqm:', features.check('raqm'))

If this prints `False`, use *Runtime → Restart session* and run the cells from the top again.

In [ ]:
!mkdir -p fonts/kannada fonts/english
!fc-list :lang=kn file | cut -d: -f1 | xargs -I{} cp {} fonts/kannada/
!for f in NotoSans-Regular NotoSans-Bold NotoSerif-Regular NotoSerif-Bold DejaVuSans DejaVuSans-Bold DejaVuSerif LiberationSans-Regular LiberationSans-Bold LiberationSerif-Regular; do find /usr/share/fonts -name "$f.ttf" -exec cp {} fonts/english/ \; ; done
!ls fonts/kannada fonts/english

Images are written to the Colab disk first, because writing thousands of small files to Drive is slow.

In [ ]:
!python src/render_chars.py --script kannada --out /content/rendered --zip data/rendered_kannada.zip
!python src/render_chars.py --script english --out /content/rendered --zip data/rendered_english.zip

In [ ]:
import csv
import random
from PIL import Image
with open('/content/rendered/kannada/labels.tsv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f, delimiter='\t'))
sheet = Image.new('L', (600, 360), 128)
for i, row in enumerate(random.Random(0).sample(rows, 60)):
    im = Image.open('/content/rendered/kannada/' + row['path'])
    im.thumbnail((56, 56))
    sheet.paste(im, ((i % 10) * 60 + 2, (i // 10) * 60 + 2))
sheet